# Datos MIT-BIH → contrato C1, por etapas

Este notebook hace la etapa de datos completa: parte de los 96 archivos crudos de `data/raw/mitbih_database/` y termina en `data/processed/beats.npz` + `meta.json`, el contrato C1 que usan Modelo y Evaluación (`docs/contracts.md`). Sigue las fases del roadmap de datos. Cada etapa explica el porqué, muestra el código y cierra con comprobaciones.

| Etapa | Qué resuelve | Qué deja definido |
|---|---|---|
| Preparación | entorno, parámetros, decisiones | `SPLIT_OF`, `check()` |
| 0 · Herramientas | leer un CSV y sus anotaciones | `read_annotations()` |
| 1 · Señal digital | muestreo, unidades, espectro | — |
| 2 · MIT-BIH por dentro | qué es un latido, clases AAMI, splits, conteos | `load_mlii()`, tabla `beats` |
| 3 · Filtrado | quitar deriva y ruido sin desfasar | `bandpass()` |
| 4 · Segmentación | ventanas, RR, ruido, bordes | `extract_windows()`, `rr_intervals()`, `mlii_noise_mask()` |
| 5 · Ensamblar C1 | juntar los 44 registros, validar, guardar | `beats.npz`, `meta.json` |
| Decisiones | las decisiones de datos (D-014 a D-019), con la evidencia de aquí | — |

**Reglas que respeta**
- Parámetros y splits salen de `ecg.config`; rutas, de `ecg.paths`. Aquí no se escribe ninguno a mano.
- `data/raw/` solo se lee.
- Solo se grafican registros de DS1 (train y val). Los de test (DS2) se procesan para construir C1 y se cuentan, pero no se miran ni se usan para decidir nada.
- Si un número no coincide con lo esperado, `check()` lo registra como discrepancia. Nada se ajusta para que cuadre.

Las cifras esperadas son las calculadas sobre estos mismos archivos el 13 oct 2026 (hallazgos del rol Datos).

## Preparación

Importaciones y estilo de las gráficas. Si `import ecg` falla, instala el paquete con `pip install -e .` desde la raíz del repo. Si ya está instalado y aun así falla, revisa que el repo no esté en una carpeta sincronizada con iCloud (Escritorio o Documentos): iCloud marca como ocultos los `.pth` del entorno y Python 3.11 los ignora. `chflags nohidden` no sirve, porque iCloud vuelve a ocultarlos en segundos; hay que mover el repo a una carpeta local y recrear el entorno.

In [ ]:
import hashlib
import json
import subprocess
from datetime import UTC, datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import signal

from ecg import config
from ecg.io import load_beats, validate_beats
from ecg.paths import PROCESSED_DIR, RAW_DIR, REPO_ROOT

# Colores fijos por papel: la misma serie lleva el mismo color en todo el notebook
C_MAIN = "#2a78d6"  # señal principal / filtrada / latidos N
C_SECOND = "#eb6834"  # comparación / latidos V / ruido
C_THIRD = "#1baf7a"  # tercera serie
C_RAW = "#9a9893"  # señal cruda como contexto
INK, MUTED = "#0b0b0b", "#52514e"

plt.rcParams.update({
    "figure.figsize": (11, 3.2),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 11,
    "lines.linewidth": 1.0,
})
pd.set_option("display.width", 120)

**Parámetros y splits.** Todo sale de `config.py` (regla 5). `SPLIT_OF` traduce registro → split. Los registros que se miran en las gráficas están en `EXPLORE`, y la aserción impide que alguien agregue uno de test por error.

In [ ]:
FS = config.FS
PRE, POST, WIN = config.WIN_PRE, config.WIN_POST, config.WIN_LEN
SPLITS = ("train", "val", "test")
AAMI_CLASSES = ["N", "S", "V", "F", "Q"]
SPLIT_OF = (
    {r: "train" for r in config.DS1_TRAIN}
    | {r: "val" for r in config.DS1_VAL}
    | {r: "test" for r in config.DS2_TEST}
)
RECORDS = sorted(SPLIT_OF)  # los 44 registros que entran a C1
EXPLORE = [101, 106, 108, 119]  # registros que se grafican: todos de DS1
assert all(SPLIT_OF[r] != "test" for r in EXPLORE), "Solo se exploran registros de DS1"

n_by_split = {s: sum(v == s for v in SPLIT_OF.values()) for s in SPLITS}
print(f"fs = {FS} Hz · ventana [r-{PRE}, r+{POST}) = {WIN} muestras")
print(f"filtro {config.BANDPASS} Hz, orden {config.FILTER_ORDER} · registros {n_by_split}")

**Decisiones de datos.** Estas constantes recogen decisiones registradas en `docs/decisions.md` el 16 oct 2026. La etapa 4 muestra cuánto cambiaría el resultado con la alternativa descartada. Cambiar un valor aquí y volver a correr todo regenera C1 con la otra opción, pero antes hay que cambiar la decisión.

In [ ]:
NOISE_CRITERION = "window"  # D-016: ruidoso si cualquier muestra de la ventana cae en un tramo `~`
RR_BEFORE_EDGE_DROP = True  # D-017: RR entre latidos AAMI consecutivos, antes de quitar bordes
DATA_VERSION = "1.0.0"  # D-014 a D-018 cerradas el 16 oct 2026

**`check()`** compara un valor con el esperado e imprime `OK` o `DIF`. Una diferencia no detiene el notebook: queda registrada en `DISCREPANCIES` para reportarla al final (regla 11). Las comparaciones con la literatura se marcan `blocking=False`, porque se espera que difieran en uno o dos latidos y lo que toca es explicarlo; las demás bloquean el guardado de C1.

In [ ]:
DISCREPANCIES: list[dict] = []


def check(name, got, expected, blocking=True):
    """Compara `got` con `expected`; si difieren, registra la discrepancia (nunca la corrige)."""
    ok = bool(got == expected)
    print(f"{'OK ' if ok else 'DIF'} · {name}: {got}" + ("" if ok else f"  (esperado: {expected})"))
    if not ok:
        DISCREPANCIES.append(
            {"check": name, "obtenido": got, "esperado": expected, "bloquea": blocking}
        )
    return ok

**Comprobaciones del entorno.** Están los 96 archivos crudos y los 44 registros de C1. La integridad byte a byte la verifica `python scripts/verify_raw_data.py` contra `data/MANIFEST.sha256`. No hay otra copia de los parámetros que vigilar: `config.py` es la única fuente (D-015).

In [ ]:
ALL_RECORDS = sorted(int(p.stem) for p in RAW_DIR.glob("*.csv"))  # los 48, excluidos incluidos
check("archivos .csv", len(ALL_RECORDS), 48)
check("archivos de anotaciones", len(list(RAW_DIR.glob("*annotations.txt"))), 48)
check("los 44 registros de C1 están en RAW_DIR", set(RECORDS) <= set(ALL_RECORDS), True)

## Etapa 0 · Herramientas: leer un registro

Un registro son 30 minutos del ECG de un paciente, en dos archivos: `101.csv` (la señal, 360 filas por segundo) y `101annotations.txt` (un evento por línea, marcado por cardiólogos). Todo el pipeline son operaciones sobre arrays, y tres de ellas cubren casi todo:

- **`shape` y `dtype`**: revísalos después de cada paso; casi todos los errores aparecen ahí primero.
- **Rebanadas `x[a:b]`**: incluyen `a` y excluyen `b`. La ventana `[r−90, r+166)` es `x[r-90:r+166]` y mide 90 + 166 = 256.
- **Máscaras booleanas**: `symbols == "N"` da un array de `True`/`False` que, usado como índice, filtra filas sin bucles.

In [ ]:
df101 = pd.read_csv(RAW_DIR / "101.csv")
print(df101.shape, list(df101.columns))
display(df101.head(3))

Los nombres de columna traen comillas simples: la columna se llama `'MLII'`, comillas incluidas. Se limpian una vez.

Los valores no están en milivoltios. Son enteros del conversor analógico-digital: el cero eléctrico es 1024 y 1 mV son 200 unidades, así que **mV = (x − 1024) / 200**. Por ejemplo, en el 119 la muestra 8178 vale 1285: (1285 − 1024) / 200 = 1,305 mV, a 8178 / 360 = 22,72 s.

Se grafica contra el tiempo en segundos, no contra el número de fila, y solo un tramo: las 650 000 muestras de una vez no dejan ver nada.

In [ ]:
df101.columns = [c.strip("'") for c in df101.columns]
x101 = (df101["MLII"].to_numpy() - config.ADC_ZERO) / config.ADC_GAIN
t101 = np.arange(len(x101)) / FS  # instante de cada muestra, en segundos

check("forma del CSV del 101", df101.shape, (650_000, 3))
print(f"MLII en mV: dtype {x101.dtype}, min {x101.min():.2f}, max {x101.max():.2f}")

fig, ax = plt.subplots()
ax.plot(t101[: 5 * FS], x101[: 5 * FS], color=C_MAIN)
ax.set(xlabel="tiempo (s)", ylabel="MLII (mV)", title="Registro 101 · primeros 5 s, sin filtrar")
plt.show()

**Anotaciones.** El archivo tiene seis campos de ancho fijo separados por espacios y luego un tabulador antes de `Aux`, que puede estar vacío. Además el encabezado `Sample #` contiene un espacio. Separar la línea entera por espacios desalinea las columnas y pierde `Aux`, así que se salta el encabezado y se separa en dos pasos: primero por el tabulador y después por espacios.

| Columna | Qué es |
|---|---|
| `sample` | muestra del CSV donde ocurre el evento (entero: es un índice) |
| `symbol` | `N`, `V`, `A`… para latidos; `+`, `~`, `"`… para otros eventos |
| `sub` | subtipo; en las marcas de ruido `~` dice qué señal está ruidosa |
| `aux` | texto libre; en las líneas `+` trae el ritmo nuevo, p. ej. `(N` o `(AFIB` |

In [ ]:
ANN_COLUMNS = ["time", "sample", "symbol", "sub", "chan", "num", "aux"]


def read_annotations(record: int) -> pd.DataFrame:
    """Lee <record>annotations.txt: una fila por anotación, en orden temporal."""
    rows = []
    with open(RAW_DIR / f"{record}annotations.txt", encoding="utf-8") as f:
        next(f)  # encabezado
        for line in f:
            fixed, _, aux = line.rstrip("\n").partition("\t")
            time, sample, symbol, sub, chan, num = fixed.split()  # falla si no son 6 campos
            rows.append((time, int(sample), symbol, int(sub), int(chan), int(num), aux.strip()))
    ann = pd.DataFrame(rows, columns=ANN_COLUMNS)
    assert ann["sample"].is_monotonic_increasing, f"{record}: anotaciones fuera de orden"
    return ann


ann101 = read_annotations(101)
print(ann101.dtypes.to_dict())
print(ann101["symbol"].value_counts().to_dict())
display(ann101.head())

El 101 es casi todo `N`. Las otras líneas no son latidos: `+` marca un cambio de ritmo (aquí, la línea inicial `(N`, sinusal), `~` un cambio en la calidad de la señal y `|` un artefacto aislado. `A` (prematuro auricular) y `Q` (no clasificable) sí son latidos.

## Etapa 1 · Señal digital: muestreo, unidades y espectro

Las decisiones de ventana y filtro (D-006, D-007) están escritas en muestras y en Hz. Esta etapa traduce entre las dos escalas.

- **Muestreo.** 360 muestras por segundo: una cada Δt = 1/360 s ≈ 2,78 ms. De tiempo a muestras se multiplica por 360.
- **Nyquist.** A 360 Hz solo se representan frecuencias hasta 180 Hz. Una componente más alta no desaparece: reaparece disfrazada de una más baja (*aliasing*). Por eso el equipo de registro filtra la señal analógica antes de digitalizarla.
- **Resolución.** Un paso del conversor vale 1/200 mV = 5 µV.

In [ ]:
dt_ms = 1000 / FS
print(f"Δt = {dt_ms:.2f} ms · ventana de {WIN} muestras = {WIN * dt_ms:.0f} ms "
      f"({PRE * dt_ms:.0f} ms antes del R y {POST * dt_ms:.0f} ms después)")
print(f"QRS de 100 ms ≈ {100 / dt_ms:.0f} muestras · Nyquist = {FS / 2:.0f} Hz · "
      f"1 paso A/D = {1000 / config.ADC_GAIN:.0f} µV")

**Frecuencia cardíaca.** El intervalo RR es el tiempo entre dos latidos consecutivos y la frecuencia es 60 / RR latidos por minuto. Se usa la mediana porque no la mueven las extrasístoles ni las pausas.

In [ ]:
r101 = ann101.loc[ann101["symbol"].isin(config.AAMI_MAP), "sample"].to_numpy()
rr101 = np.diff(r101) / FS
print(f"RR mediano del 101: {np.median(rr101):.3f} s → {60 / np.median(rr101):.0f} lpm")

marks = r101[r101 < 10 * FS]
fig, ax = plt.subplots()
ax.plot(t101[: 10 * FS], x101[: 10 * FS], color=C_MAIN, label="MLII")
ax.plot(marks / FS, x101[marks], "o", ms=7, mfc="none", mew=1.2, color=C_SECOND,
        label="anotación de latido")
ax.set(xlabel="tiempo (s)", ylabel="MLII (mV)", title="Registro 101 · 10 s con sus anotaciones")
ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
plt.show()

**Espectro (PSD de Welch).** Toda señal se puede descomponer en senoidales; la densidad espectral de potencia dice cuánta energía hay en cada frecuencia. El método de Welch corta la señal en trozos de `nperseg` muestras, calcula el espectro de cada uno y los promedia. Trozos más largos dan más resolución en frecuencia, pero se promedian menos y la curva sale más ruidosa. Se grafica con eje y logarítmico porque la potencia cae varios órdenes de magnitud; en escala lineal todo lo que no sea baja frecuencia parecería cero. Sin `fs=360`, el eje x saldría en ciclos por muestra y no en Hz.

In [ ]:
def plot_band_marks(ax):
    """Líneas de referencia del filtro del proyecto y de la red eléctrica (60 Hz)."""
    marks = [(config.BANDPASS[0], "0,5 Hz", 1.0), (config.BANDPASS[1], "40 Hz", 1.0),
             (60, "60 Hz", 0.9)]
    for f0, txt, height in marks:  # 60 Hz más abajo para no chocar con 40 Hz en escala log
        ax.axvline(f0, color=MUTED, ls="--", lw=0.8)
        ax.annotate(f" {txt}", (f0, height), xycoords=("data", "axes fraction"), va="top",
                    color=MUTED, fontsize=9)


fig, ax = plt.subplots(figsize=(11, 3.6))
for nperseg, color, z in [(512, C_MAIN, 3), (8192, C_SECOND, 2)]:
    f, pxx = signal.welch(x101, fs=FS, nperseg=nperseg)
    ax.semilogy(f, pxx, color=color, zorder=z, label=f"nperseg = {nperseg} ({nperseg / FS:.1f} s)")
plot_band_marks(ax)
ax.set(xlabel="frecuencia (Hz)", ylabel="PSD (mV²/Hz)", xlim=(0, FS / 2),
       title="Registro 101 · PSD de MLII sin filtrar")
ax.legend(loc="lower left")
plt.show()

**Qué se ve.**
- La potencia se concentra por debajo de ~20 Hz, donde viven las ondas P, QRS y T, y cae varios órdenes hacia 180 Hz.
- Por debajo de 0,5 Hz hay sobre todo deriva de la línea base (respiración, movimiento).
- El 101 sí tiene interferencia de la red: un pico estrecho en 60 Hz y sus armónicos en 120 y 180 Hz.
- Con trozos largos (naranja) esos picos salen más altos y estrechos, porque la resolución es mayor. Con trozos cortos (azul) se ensanchan y la curva sale más suave.
- El ruido muscular es de banda ancha y se solapa con el QRS, así que ningún filtro lo quita del todo.

El filtro del proyecto conserva la franja entre las líneas de 0,5 y 40 Hz.

## Etapa 2 · MIT-BIH por dentro: anotaciones, clases y splits

Aquí se decide qué cuenta como latido. Un error en esta etapa cambia los conteos de todo el proyecto, y la tabla publicada permite detectarlo.

- **Latidos y no-latidos.** Solo los 15 símbolos de `config.AAMI_MAP` son latidos y entran a C1. El resto describe ritmo (`+`), calidad de señal (`~`), comentarios (`"`), ondas P no conducidas (`x`), artefactos (`|`), inicio y fin de fibrilación ventricular (`[` `]`) y ondas de aleteo ventricular (`!`). Es la práctica habitual con AAMI (D-014).
- **Clases AAMI** (norma EC57), para comparar con la literatura: **N** (N, L, R, e, j) · **S** supraventricular (A, a, J, S) · **V** ventricular (V, E) · **F** fusión (F) · **Q** desconocido o con marcapasos (/, f, Q). Además, el proyecto usa `label = 0` solo para el símbolo `N` estricto (D-005).
- **Símbolo ≠ ritmo.** El símbolo describe el origen y la forma del QRS. Un latido con QRS normal durante una fibrilación auricular sigue siendo `N`, aunque no tenga onda P. El ritmo va aparte, en las líneas `+`.
- **Split inter-paciente.** La forma del latido es muy personal: si un paciente queda en train y en test, el modelo lo reconoce en lugar de generalizar y las métricas se inflan. Se usa la partición de de Chazal (D-003) tal cual, aunque 201 (train) y 202 (test) son del mismo paciente (D-013).

Se leen las anotaciones de los 48 registros. Para contar no hace falta la señal.

In [ ]:
ann = pd.concat([read_annotations(r).assign(record=r) for r in ALL_RECORDS], ignore_index=True)
is_beat = ann["symbol"].isin(config.AAMI_MAP)
print(f"{len(ann):,} anotaciones en {ann['record'].nunique()} registros; "
      f"{is_beat.sum():,} son latidos AAMI")

**No-latidos.** Conteo en los 48 registros, contra los hallazgos. Las 472 ondas `!` están todas en el 207 (train): son ondas de aleteo, no complejos QRS que se puedan clasificar.

In [ ]:
EXPECTED_NON_BEATS = {"+": 1291, "~": 616, "!": 472, '"': 437, "x": 193, "|": 132, "[": 6, "]": 6}
non_beats = ann.loc[~is_beat, "symbol"].value_counts().to_dict()
print(non_beats)
check("no-latidos por símbolo", non_beats, EXPECTED_NON_BEATS)
check("registros con ondas '!'", ann.loc[ann["symbol"] == "!", "record"].unique().tolist(), [207])
check("símbolos fuera de AAMI_MAP que no son los 8 conocidos",
      sorted(set(non_beats) - set(EXPECTED_NON_BEATS)), [])

**¿Dónde está MLII?** Leer la derivación por posición es un error silencioso: en el 114 la primera columna es V5 y no saltaría ningún aviso. Basta con leer el encabezado de los 48 CSV.

In [ ]:
headers = {}
for r in ALL_RECORDS:
    with open(RAW_DIR / f"{r}.csv", encoding="utf-8") as f:
        headers[r] = [c.strip("'") for c in f.readline().strip().split(",")][1:]
not_first = {r: leads for r, leads in headers.items() if leads[0] != config.LEAD}
print(not_first)
check("registros donde MLII no es la 1.ª derivación", sorted(not_first), [102, 104, 114])
check("102 y 104 (sin MLII) están excluidos", {102, 104} <= set(config.EXCLUDED), True)

`load_mlii` elige la columna **por nombre** y falla con un mensaje claro si no existe. Devuelve también la posición de MLII entre las dos derivaciones (0 o 1), que hace falta para leer las marcas de ruido en la etapa 4.

In [ ]:
def load_mlii(record: int) -> tuple[np.ndarray, int]:
    """Devuelve (MLII en mV, posición de MLII entre las dos derivaciones del registro)."""
    df = pd.read_csv(RAW_DIR / f"{record}.csv")
    leads = [c.strip("'") for c in df.columns[1:]]
    if config.LEAD not in leads:
        raise ValueError(f"El registro {record} no tiene {config.LEAD}; trae {leads}")
    channel = leads.index(config.LEAD)
    adc = df.iloc[:, 1 + channel].to_numpy()
    return (adc - config.ADC_ZERO) / config.ADC_GAIN, channel


x114, ch114 = load_mlii(114)
third_col_114 = pd.read_csv(RAW_DIR / "114.csv").iloc[:, 2].to_numpy()
check("114: MLII es la derivación 1", ch114, 1)
check("114: devuelve la 3.ª columna del archivo (V5 es la 2.ª)",
      bool(np.array_equal(x114, (third_col_114 - config.ADC_ZERO) / config.ADC_GAIN)), True)
try:
    load_mlii(102)
except ValueError as e:
    print("102 →", e)

**Tabla de latidos.** Una fila por latido de los 44 registros, con su clase AAMI, su split y su etiqueta. Es la base de todos los conteos. Los conteos son "antes de descartar bordes": los latidos cuya ventana no cabe en el registro se quitan en la etapa 4.

In [ ]:
beats = ann[is_beat & ann["record"].isin(SPLIT_OF)].copy()
beats["aami"] = beats["symbol"].map(config.AAMI_MAP)
beats["split"] = beats["record"].map(SPLIT_OF)
beats["label"] = (~beats["symbol"].isin(config.NORMAL_SYMBOLS)).astype(int)

counts = pd.crosstab(beats["split"], beats["aami"], margins=True, margins_name="total")
display(counts.reindex(index=[*SPLITS, "total"], columns=[*AAMI_CLASSES, "total"]))

by_split = beats.groupby("split").size()
n_strict = beats[beats["symbol"] == "N"].groupby("split").size()
check("latidos AAMI en train", int(by_split["train"]), 41_012)
check("latidos AAMI en val", int(by_split["val"]), 10_009)
check("latidos AAMI en test", int(by_split["test"]), 49_712)
check("N estricto en train", int(n_strict["train"]), 29_607)
check("N estricto en val", int(n_strict["val"]), 8_495)

**Comparación con la literatura.** DS1 = train + val. La referencia (BioMed Eng OnLine 13:90, 2014, tabla 1) reporta 51 020 latidos en DS1 y 49 711 en DS2; por clase, DS1 tiene N 45 868 · S 942 · V 3 787 · F 415 · Q 8.

In [ ]:
REF_DS1 = {"N": 45_868, "S": 942, "V": 3_787, "F": 415, "Q": 8}
REF_DS2_TOTAL = 49_711

ds1 = beats[beats["split"] != "test"]
lit = pd.DataFrame({"este notebook": ds1["aami"].value_counts(), "referencia": pd.Series(REF_DS1)})
lit = lit.loc[AAMI_CLASSES]
lit.loc["total"] = lit.sum()
lit["diferencia"] = lit["este notebook"] - lit["referencia"]
display(lit)

check("DS1 total vs literatura", len(ds1), sum(REF_DS1.values()), blocking=False)
check("DS2 total vs literatura", int(by_split["test"]), REF_DS2_TOTAL, blocking=False)
check("DS1 por clase vs literatura", ds1["aami"].value_counts().to_dict(), REF_DS1, blocking=False)

**Lectura.** Los totales difieren en un latido, tanto en DS1 como en DS2. En DS1 la diferencia no está en una sola clase: se reparte en N −2, S +2 y V +1. No se ajusta nada. Para explicarla hay que comparar registro por registro con las tablas del directorio de PhysioNet. Una hipótesis por verificar es que el export CSV venga de una revisión de las anotaciones distinta de la que usó la referencia. Queda en el resumen de discrepancias.

**Detalle por símbolo.** Val tiene muy pocos latidos de algunas clases: 14 F, ningún Q y ningún bloqueo de rama (L, R). Las métricas por clase en val van a ser frágiles.

In [ ]:
sym_split = pd.crosstab(beats["symbol"], beats["split"]).reindex(columns=list(SPLITS))
display(sym_split.T)
check("F en val", int(sym_split.loc["F", "val"]), 14)
check("Q (clase AAMI) en val", int((ds1["aami"] == "Q").groupby(ds1["split"]).sum()["val"]), 0)
check("L y R en val", int(sym_split.loc[["L", "R"], "val"].sum()), 0)

**El ritmo de cada latido.** Cada línea `+` abre un ritmo que dura hasta la siguiente `+`, como un interruptor. Para saber en qué ritmo cae un latido se busca la última `+` anterior con `np.searchsorted`. `state_at` hace eso de forma genérica, y en la etapa 4 se reutiliza para el ruido.

In [ ]:
def state_at(event_samples, event_values, samples, default):
    """Valor del último evento ocurrido en o antes de cada muestra; `default` antes del primero."""
    event_values = np.asarray(event_values)
    if len(event_samples) == 0:
        return np.full(len(samples), default)
    idx = np.searchsorted(event_samples, samples, side="right") - 1
    return np.where(idx >= 0, event_values[np.clip(idx, 0, None)], default)


def rhythm_of(record_ann: pd.DataFrame, samples: np.ndarray) -> np.ndarray:
    """Ritmo (texto de Aux de la última línea '+') en cada muestra."""
    rh = record_ann[record_ann["symbol"] == "+"]
    return state_at(rh["sample"].to_numpy(), rh["aux"].to_numpy(), samples, default="")


beats["rhythm"] = ""
for r, g in beats.groupby("record"):
    beats.loc[g.index, "rhythm"] = rhythm_of(ann[ann["record"] == r], g["sample"].to_numpy())

n_train = beats[(beats["split"] == "train") & (beats["symbol"] == "N")]
display(n_train["rhythm"].value_counts().to_frame("latidos N en train").T)
not_sinus = n_train["rhythm"].isin(["(AFIB", "(AFL", "(PREX"])
print(f"N de train en FA, flutter o preexcitación: {not_sinus.sum():,} de {len(n_train):,} "
      f"({not_sinus.mean():.1%})")
print(n_train[not_sinus].groupby(["record", "rhythm"]).size().to_dict())

Cerca del 15 % de los N de train no ocurren en ritmo sinusal: hay fibrilación auricular en el 201 y el 203, flutter en el 203 y preexcitación en el 230. El autoencoder "de latidos normales" aprende también esas formas. D-018 mantiene como normal el símbolo N en cualquier ritmo, por comparabilidad con la literatura, y registra esta proporción como limitación. C1 no guarda el ritmo; este notebook lo calcula aparte.

## Etapa 3 · Filtrado: limpiar sin deformar

El autoencoder aprende formas. Si la línea base sube y baja o hay ruido de red, aprende ruido; pero un mal filtro deforma justo lo que se quiere estudiar.

- **Pasa-banda 0,5–40 Hz** (D-007): quita la deriva lenta por debajo de 0,5 Hz y el ruido rápido por encima de 40 Hz.
- **Butterworth de orden n**: es plano en la banda de paso, así que no altera la amplitud de lo que conserva. Su pasa-bajas analógico cumple |H(f)|² = 1 / (1 + (f/fc)^(2n)). En la frecuencia de corte la ganancia es 1/√2 ≈ 0,707, es decir, −3 dB.
- **Fase cero con `sosfiltfilt`**: un filtro retrasa cada frecuencia de forma distinta y eso corre y deforma las ondas. Filtrar hacia adelante y luego hacia atrás cancela el retraso y aplica la atenuación dos veces, así que la ganancia total es |H|².
- **Forma SOS**: el filtro como cadena de secciones de segundo orden, numéricamente más estable que los coeficientes `b, a`.
- **Bordes**: al inicio y al final de la señal el filtro no tiene historia y produce transitorios. Por eso se filtra el registro completo y después se recorta cada latido.
- **Criterio clínico**: el ECG diagnóstico usa un pasa-altas de 0,05 Hz para no deformar el segmento ST. Las recomendaciones de la AHA (Kligfield 2007) permiten subirlo hasta ~0,67 Hz si el filtro es lineal y de fase cero, como este. El corte de 40 Hz es propio de los monitores y suaviza un poco el pico del QRS.

In [ ]:
SOS = signal.butter(config.FILTER_ORDER, config.BANDPASS, btype="bandpass", fs=FS, output="sos")


def bandpass(x: np.ndarray) -> np.ndarray:
    """Pasa-banda Butterworth de fase cero (D-007), aplicado a una señal continua completa."""
    return signal.sosfiltfilt(SOS, x)


freqs = np.geomspace(0.01, FS / 2, 4000)
_, h = signal.sosfreqz(SOS, worN=freqs, fs=FS)
gain_db = 20 * np.log10(np.abs(h))

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.semilogx(freqs, gain_db, color=C_MAIN, lw=2, label="una pasada: |H|")
ax.semilogx(freqs, 2 * gain_db, color=C_SECOND, lw=2, label="sosfiltfilt: |H|²")
plot_band_marks(ax)
ax.axhline(-3, color=MUTED, lw=0.6)
ax.set(xlabel="frecuencia (Hz, escala log)", ylabel="ganancia (dB)", ylim=(-80, 5),
       title=f"Respuesta en frecuencia · Butterworth orden {config.FILTER_ORDER}, "
             f"{len(SOS)} secciones SOS")
ax.legend(loc="lower center")
plt.show()

**Ejemplo resuelto.** Pasa-bajas analógico con n = 3 y fc = 40 Hz. En 40 Hz: |H|² = 1/(1 + 1) = 0,5, así que |H| = 0,707. En 80 Hz: (80/40)⁶ = 64, |H|² = 1/65 y |H| ≈ 0,124 (−18 dB).

El filtro digital atenúa **más** que esa fórmula por encima del corte. `butter(..., fs=360)` usa la transformación bilineal, que comprime todo el eje de frecuencias en 0–180 Hz. La fórmula vuelve a cuadrar si cada frecuencia f se reemplaza por su equivalente en ese eje comprimido, tan(πf/fs). La tabla compara las tres versiones.

In [ ]:
def butter_lowpass_gain(f, fc=config.BANDPASS[1], n=config.FILTER_ORDER, bilinear=False):
    """|H| del pasa-bajas Butterworth analógico; con bilinear=True, en el eje comprimido."""
    ratio = np.tan(np.pi * f / FS) / np.tan(np.pi * fc / FS) if bilinear else f / fc
    return 1 / np.sqrt(1 + ratio ** (2 * n))


probe = np.array([40.0, 60.0, 80.0])
_, h_probe = signal.sosfreqz(SOS, worN=probe, fs=FS)
gains = pd.DataFrame({
    "fórmula analógica": butter_lowpass_gain(probe),
    "fórmula, eje bilineal": butter_lowpass_gain(probe, bilinear=True),
    "digital, una pasada": np.abs(h_probe),
    "digital, sosfiltfilt": np.abs(h_probe) ** 2,
}, index=pd.Index(probe, name="Hz"))
display(gains.round(3))
print(f"Atenuación total a 60 Hz con sosfiltfilt: {20 * np.log10(np.abs(h_probe[1]) ** 2):.1f} dB")

A 60 Hz la fórmula analógica da 0,284 y el filtro real 0,238. La diferencia la explica la compresión del eje: con el eje bilineal la fórmula da prácticamente lo mismo que el filtro. Con `sosfiltfilt` la red eléctrica queda en ~5,7 % de su amplitud (unos −25 dB). En la PSD del 108 filtrado, más abajo, el pico de 60 Hz sigue visible pero varios órdenes de magnitud por debajo de la potencia del ECG, así que no hace falta un filtro notch adicional.

**Senoidales de prueba.** Se filtran senoidales puras de 0,1, 10 y 60 Hz (60 s cada una) y se mide su amplitud lejos de los bordes. Lo medido debe coincidir con |H|².

La amplitud se mide como √2 · RMS y no como (máximo − mínimo) / 2. A 360 Hz una senoidal de 60 Hz tiene solo 6 muestras por ciclo, ninguna cae justo en el pico, y el máximo muestreado vale sin(60°) ≈ 0,866 de la amplitud real. La primera versión de este notebook medía con el máximo y por eso marcaba una discrepancia a 60 Hz: el error estaba en la medición, no en el filtro.

In [ ]:
tt = np.arange(60 * FS) / FS
core = slice(10 * FS, 50 * FS)  # 10 s de margen en cada borde; 40 s = ciclos enteros
rows = []
for f0 in (0.1, 10.0, 60.0):
    y = bandpass(np.sin(2 * np.pi * f0 * tt))
    _, h0 = signal.sosfreqz(SOS, worN=[f0], fs=FS)
    rows.append((f0, np.sqrt(2 * np.mean(y[core] ** 2)), np.abs(h0[0]) ** 2))
sines = pd.DataFrame(rows, columns=["Hz", "amplitud medida", "|H|² esperado"]).set_index("Hz")
display(sines.round(4))
check("10 Hz pasa casi intacta (> 0,99)", bool(sines.loc[10.0, "amplitud medida"] > 0.99), True)
check("0,1 y 60 Hz salen atenuadas (< 0,1)",
      bool((sines.loc[[0.1, 60.0], "amplitud medida"] < 0.1).all()), True)
check("medido ≈ |H|² (error < 0,005)",
      bool(np.allclose(sines["amplitud medida"], sines["|H|² esperado"], atol=0.005)), True)

**Registro 108, crudo contra filtrado.** El 108 (train) tiene mucha deriva de base y ruido. En este tramo, cerca del minuto 20, la línea base cae y sube varios mV en pocos segundos.

El filtro devuelve a cero la deriva lenta y deja los QRS en su sitio. Pero el salto brusco cerca de 1217 s tiene energía por encima de 0,5 Hz y deja un residuo de casi 1 mV. Ningún pasa-banda puede quitar un artefacto que comparte frecuencias con el ECG: para eso están las marcas de ruido. Aquí las anotaciones marcan MLII como ruidosa desde 1215,4 s (`~` con subtipo 3), así que esos latidos llevarán `mlii_noisy = True` (etapa 4).

In [ ]:
x108, ch108 = load_mlii(108)
y108 = bandpass(x108)

seg = slice(1213 * FS, 1221 * FS)
t108 = np.arange(len(x108)) / FS
fig, ax = plt.subplots()
ax.plot(t108[seg], x108[seg], color=C_RAW, label="cruda")
ax.plot(t108[seg], y108[seg], color=C_MAIN, label="filtrada 0,5–40 Hz")
ax.axhline(0, color=MUTED, lw=0.6)
ax.set(xlabel="tiempo (s)", ylabel="MLII (mV)", title="Registro 108 · 8 s con deriva de base")
ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
plt.show()

fig, ax = plt.subplots(figsize=(11, 3.6))
for x, color, label in [(x108, C_RAW, "cruda"), (y108, C_MAIN, "filtrada")]:
    f, pxx = signal.welch(x, fs=FS, nperseg=4096)
    ax.semilogy(f, pxx, color=color, label=label)
plot_band_marks(ax)
ax.set(xlabel="frecuencia (Hz)", ylabel="PSD (mV²/Hz)", xlim=(0, FS / 2), ylim=(1e-12, None),
       title="Registro 108 · PSD antes y después del filtro")
ax.legend(loc="lower left")
plt.show()

**Fase: por qué `sosfiltfilt` y no `sosfilt`.** Se toma un latido del 101 y se compara la señal cruda con las dos versiones filtradas. `sosfilt` (solo hacia adelante) retrasa el QRS varias muestras y lo deforma; `sosfiltfilt` lo deja en su sitio. La señal cruda aparece desplazada en vertical porque conserva la deriva de base.

In [ ]:
y101 = bandpass(x101)
forward_only = signal.sosfilt(SOS, x101)

r0 = r101[100]
w = slice(r0 - PRE, r0 + POST)
t_ms = (np.arange(WIN) - PRE) * dt_ms
fig, ax = plt.subplots()
ax.plot(t_ms, x101[w], color=C_RAW, label="cruda")
ax.plot(t_ms, forward_only[w], color=C_SECOND, label="sosfilt (solo ida)")
ax.plot(t_ms, y101[w], color=C_MAIN, label="sosfiltfilt (ida y vuelta)")
ax.axvline(0, color=MUTED, ls="--", lw=0.8)
ax.set(xlabel="ms desde la anotación", ylabel="MLII (mV)",
       title=f"Registro 101 · latido en la muestra {r0}")
ax.legend(loc="upper right")
plt.show()

lag = {name: int(np.argmax(sig[w]) - np.argmax(x101[w]))
       for name, sig in [("sosfilt", forward_only), ("sosfiltfilt", y101)]}
print(f"Desplazamiento del máximo respecto de la señal cruda (muestras): {lag}")

**¿Se mueve el pico R?** Para cada latido del 101 se busca el máximo dentro de ±10 muestras alrededor de la anotación, en la señal cruda y en la filtrada. Un pulso sintético simétrico sirve de control: con fase cero, su máximo no se puede mover.

In [ ]:
inner = r101[(r101 >= 10) & (r101 < len(x101) - 10)]
around = inner[:, None] + np.arange(-10, 11)
shift = np.argmax(y101[around], axis=1) - np.argmax(x101[around], axis=1)
display(pd.Series(shift).value_counts().sort_index().to_frame("latidos").T)
print(f"El máximo queda en la misma muestra en el {np.mean(shift == 0):.1%} de los latidos")

pulse = np.exp(-0.5 * ((np.arange(4000) - 2000) / 6) ** 2)
check("pulso simétrico: el máximo sigue en la muestra 2000", int(np.argmax(bandpass(pulse))), 2000)
check("bandpass conserva el largo", len(y101), len(x101))

En la gran mayoría de latidos el máximo no se mueve y en el resto se corre ±1–2 muestras (3–6 ms). No es desfase: el pulso de control queda exacto. Lo que pasa es que el filtro quita la pendiente de la línea base y suaviza un poco el pico, y en un R casi plano eso basta para que el máximo cambie de muestra. A C1 no le afecta, porque la ventana se ancla en la anotación y no en el máximo.

## Etapa 4 · Segmentación, intervalos RR y ruido

Esta etapa crea las filas de C1. Cada decisión que se toma aquí (qué latidos, qué ventana, cómo se mide el RR, qué es ruido) termina siendo una columna del contrato.

- **Punto fiducial.** La anotación marca el QRS, pero no siempre su máximo: en el 119 el N está anotado en 8178 y el máximo está en 8179. Al autoencoder le basta con que la referencia sea consistente, así que la ventana se ancla en la anotación.
- **Ventana asimétrica [r−90, r+166)** (D-006): 250 ms antes y 461 ms después, porque la onda T, que va después del QRS, dura más que la P. ¿Cabe la P? Empieza un intervalo PR (~200 ms) más ~40 ms antes del pico R: 240 ms ≈ 86 muestras, menos que 90. Con un bloqueo AV de primer grado (PR = 300 ms) son 340 ms ≈ 122 muestras y la P queda fuera.
- **Ventanas que se solapan.** Si el latido siguiente llega antes de 461 ms (unos 130 lpm), su QRS entra en la ventana. Es normal: cada fila guarda su ventana completa.
- **RR.** `rr_prev = (r − r_anterior) / 360` y `rr_next = (r_siguiente − r) / 360`, en segundos. El primer latido de un registro no tiene anterior (`NaN`) y el último no tiene siguiente.
- **Ruido como interruptor.** Cada `~` fija el estado hasta la siguiente `~`. El subtipo es un campo de bits: el bit 0 enciende la señal 0 y el bit 1 la señal 1. MLII es la señal 0 en todos los registros salvo el 114, donde es la 1. El subtipo −1 tiene todos los bits encendidos (complemento a dos), así que marca ambas señales. Un latido es ruidoso si **cualquier muestra de su ventana** cae en un tramo ruidoso (D-016).
- **Bordes.** Si `r − 90 < 0` o `r + 166 > 650 000`, la ventana se sale del registro y el latido se descarta.

In [ ]:
def extract_windows(x: np.ndarray, r: np.ndarray, pre: int = PRE, post: int = POST):
    """Recorta x[r-pre : r+post] para cada r. Devuelve (ventanas, keep); los bordes se descartan."""
    keep = (r - pre >= 0) & (r + post <= len(x))
    idx = r[keep, None] + np.arange(-pre, post)
    return x[idx], keep


def rr_intervals(r: np.ndarray, fs: int = FS) -> tuple[np.ndarray, np.ndarray]:
    """RR previo y siguiente de cada latido, en segundos; NaN donde no hay vecino."""
    d = np.diff(r) / fs
    return np.r_[np.nan, d], np.r_[d, np.nan]


def _mlii_noise_events(record_ann: pd.DataFrame, channel: int):
    noise = record_ann[record_ann["symbol"] == "~"]
    bit_on = ((noise["sub"].to_numpy() >> channel) & 1).astype(bool)  # -1 >> k sigue siendo -1
    return noise["sample"].to_numpy(), bit_on


def mlii_noise_mask(record_ann: pd.DataFrame, r: np.ndarray, channel: int) -> np.ndarray:
    """D-016, criterio "r_sample": True si la muestra del R cae en un tramo con MLII ruidosa."""
    s, bit_on = _mlii_noise_events(record_ann, channel)
    return state_at(s, bit_on, r, default=False).astype(bool)


def mlii_noise_mask_window(record_ann, r, channel, pre=PRE, post=POST) -> np.ndarray:
    """D-016, criterio "window": True si cualquier muestra de la ventana cae en un tramo ruidoso."""
    s, bit_on = _mlii_noise_events(record_ann, channel)
    noisy_at_start = state_at(s, bit_on, r - pre, default=False).astype(bool)
    lo = np.searchsorted(s, r - pre, side="right")  # marcas dentro de (r-pre, r+post-1]
    hi = np.searchsorted(s, r + post - 1, side="right")
    noisy_marks = np.r_[0, np.cumsum(bit_on)]
    return noisy_at_start | (noisy_marks[hi] - noisy_marks[lo] > 0)


NOISE_FN = {"r_sample": mlii_noise_mask, "window": mlii_noise_mask_window}[NOISE_CRITERION]

**Casos de prueba.** Son los mismos que irán a `tests/` cuando estas funciones pasen a `src/ecg/data/`.

In [ ]:
_, keep = extract_windows(np.zeros(1000), np.array([50, 500, 900]))
check("r=50 y r=900 se descartan por borde (largo 1000)", keep.tolist(), [False, True, False])
win, _ = extract_windows(np.arange(1000.0), np.array([500]))
check("la ventana mide 256 y empieza en r-90", (win.shape[1], float(win[0, 0])), (256, 410.0))

prev, nxt = rr_intervals(np.array([100, 460, 820]))
check("rr_prev de R = [100, 460, 820]", [str(v) for v in prev], ["nan", "1.0", "1.0"])
check("rr_next de R = [100, 460, 820]", [str(v) for v in nxt], ["1.0", "1.0", "nan"])

toy = pd.DataFrame({"symbol": ["~", "~", "~"], "sample": [100, 200, 300], "sub": [1, 2, 0]})
r_toy = np.array([50, 150, 250, 350])
check("sub=1 marca la señal 0 (MLII en casi todos)",
      mlii_noise_mask(toy, r_toy, 0).tolist(), [False, True, False, False])
check("sub=1 no marca MLII en un registro tipo 114 (canal 1)",
      mlii_noise_mask(toy, r_toy, 1).tolist(), [False, False, True, False])
toy_minus1 = toy.assign(sub=[-1, 0, 0])
check("sub=-1 marca MLII en ambos canales",
      [mlii_noise_mask(toy_minus1, r_toy, ch).tolist() for ch in (0, 1)],
      [[False, True, False, False]] * 2)
check("criterio ventana: un R limpio con ruido 40 muestras después cuenta como ruidoso",
      mlii_noise_mask_window(toy, np.array([60]), 0).tolist(), [True])

**El ejemplo de referencia: registro 119 (val).** En este tramo de bigeminismo cada N va seguido de un V. El N de la muestra 8178 y el V de la 8377 tienen que dar exactamente los valores de la tabla del roadmap. El V llega solo 199 muestras (0,55 s) después del N, menos que la ventana: por eso las dos ventanas se solapan.

In [ ]:
x119, ch119 = load_mlii(119)
y119 = bandpass(x119)
ann119 = read_annotations(119)
b119 = ann119[ann119["symbol"].isin(config.AAMI_MAP)].reset_index(drop=True)
r119 = b119["sample"].to_numpy()
prev119, next119 = rr_intervals(r119)
noisy119 = NOISE_FN(ann119, r119, ch119)

ref = {8178: ("N", 1.303, 0.553), 8377: ("V", 0.553, 1.342)}
for sample, (sym, rr_p, rr_n) in ref.items():
    i = int(np.flatnonzero(r119 == sample)[0])
    got = (b119["symbol"][i], round(prev119[i], 3), round(next119[i], 3), bool(noisy119[i]))
    check(f"119 · latido {sample} (símbolo, rr_prev, rr_next, mlii_noisy)", got,
          (sym, rr_p, rr_n, False))
check("119 · el máximo del N 8178 está en la muestra", 8170 + int(np.argmax(x119[8170:8190])), 8179)

t0, t1 = int(22.2 * FS), int(27.2 * FS)
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.axvspan((8377 - PRE) / FS, (8377 + POST) / FS, color=C_SECOND, alpha=0.15,
           label="ventana del V 8377")
ax.axvspan((8178 - PRE) / FS, (8178 + POST) / FS, fill=False, ec=C_MAIN, ls="--", lw=1.2,
           label="ventana del N 8178")
ax.plot(np.arange(t0, t1) / FS, y119[t0:t1], color=INK, lw=1.1)
for r_ in r119[(r119 >= t0) & (r119 < t1)]:
    sym = b119["symbol"][int(np.flatnonzero(r119 == r_)[0])]
    ax.annotate(f"{sym} {r_}", (r_ / FS, 1.0), xycoords=("data", "axes fraction"),
                ha="center", va="bottom", fontsize=9, color=INK)
ax.set(xlabel="tiempo (s)", ylabel="MLII filtrada (mV)")
ax.set_title("Registro 119 · bigeminismo: cada N seguido de un V", pad=18)
ax.legend(loc="lower right")
plt.show()

El 119 también trae un tramo con `~` de subtipo 2 desde el minuto 20:06: es ruidosa la señal 1 (V1), no MLII. Los latidos de ese tramo deben quedar con `mlii_noisy = False`.

In [ ]:
display(ann119.loc[ann119["symbol"] == "~", ["time", "sample", "sub"]])
in_v1_noise = (r119 >= 434_464) & (r119 < 440_729)
print(f"latidos entre 20:06 y 20:24: {in_v1_noise.sum()}")
check("119 · latidos del tramo sub=2 con mlii_noisy", int(noisy119[in_v1_noise].sum()), 0)

**N contra V en un registro de train.** 50 latidos de cada clase del 106, elegidos a intervalos regulares (no al azar, para que la figura sea reproducible) y alineados en la anotación. Qué se ve:
- Los N son casi idénticos entre sí, con onda P (hacia −100 ms), QRS estrecho y T.
- Los V no tienen onda P, su QRS es más ancho y su forma varía mucho de un latido a otro.
- Como el V es prematuro, el inicio de su ventana todavía contiene la onda T del latido anterior.

Esa diferencia de forma es la que el autoencoder tiene que reconstruir mal.

In [ ]:
x106, _ = load_mlii(106)
y106 = bandpass(x106)
ann106 = read_annotations(106)
b106 = ann106[ann106["symbol"].isin(config.AAMI_MAP)]
fig, ax = plt.subplots(figsize=(11, 3.8))
for sym, color in [("N", C_MAIN), ("V", C_SECOND)]:
    r_sym = b106.loc[b106["symbol"] == sym, "sample"].to_numpy()
    r_sym = r_sym[np.linspace(0, len(r_sym) - 1, 50).astype(int)]
    win, _ = extract_windows(y106, r_sym)
    ax.plot(t_ms, win.T, color=color, alpha=0.12, lw=0.8)
    ax.plot(t_ms, win.mean(axis=0), color=color, lw=2.2, label=f"{sym} (media de 50)")
ax.axvline(0, color=MUTED, ls="--", lw=0.8)
ax.set(xlabel="ms desde la anotación", ylabel="MLII filtrada (mV)",
       title="Registro 106 · latidos N y V superpuestos")
ax.legend(loc="upper right")
plt.show()

**RR por clase en DS1.** Se calculan con las anotaciones (no hace falta la señal), registro por registro, para no unir el último latido de un paciente con el primero del siguiente. Los S y los V son prematuros, así que su `rr_prev` es más corto que el de los N.

In [ ]:
beats["rr_prev"], beats["rr_next"] = np.nan, np.nan
for r, g in beats.groupby("record"):
    beats.loc[g.index, "rr_prev"], beats.loc[g.index, "rr_next"] = rr_intervals(
        g["sample"].to_numpy()
    )

ds1 = beats[beats["split"] != "test"]
bins = np.linspace(0, 2.0, 61)
fig, axes = plt.subplots(1, 5, figsize=(13, 2.8), sharex=True)
for ax, cls in zip(axes, AAMI_CLASSES):
    rr = ds1.loc[ds1["aami"] == cls, "rr_prev"].dropna()
    ax.hist(rr, bins=bins, color=C_MAIN)
    ax.axvline(rr.median(), color=INK, lw=1)
    ax.set_title(f"{cls} · n = {len(rr):,} · mediana {rr.median():.2f} s", fontsize=9)
    ax.set_xlabel("rr_prev (s)")
axes[0].set_ylabel("latidos")
fig.suptitle("DS1 · intervalo RR previo por clase AAMI", y=1.04)
plt.show()

**Tramos ruidosos del 108.** El registro completo, reducido a una de cada 20 muestras solo para graficar, con los tramos en que MLII está marcada como ruidosa.

In [ ]:
ann108 = read_annotations(108)
noise_s, noise_on = _mlii_noise_events(ann108, ch108)
ends = np.r_[noise_s[1:], len(x108)]
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(t108[::20], y108[::20], color=C_MAIN, lw=0.4, label="MLII filtrada")
label = "MLII ruidosa (~)"
for start, end, on in zip(noise_s, ends, noise_on):
    if on:
        ax.axvspan(start / FS, end / FS, color=C_SECOND, alpha=0.25, lw=0, label=label)
        label = None  # una sola entrada en la leyenda
ax.set(xlabel="tiempo (s)", ylabel="mV", xlim=(0, len(x108) / FS),
       title="Registro 108 completo · tramos con MLII ruidosa")
ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
plt.show()

b108 = ann108.loc[ann108["symbol"].isin(config.AAMI_MAP), "sample"].to_numpy()
print(f"108: {NOISE_FN(ann108, b108, ch108).sum()} de {len(b108)} latidos con MLII ruidosa")

**Bordes.** Se cuenta cuántos latidos se descartan por registro porque su ventana no cabe. Basta con las anotaciones: todos los registros miden 650 000 muestras.

In [ ]:
N_SAMPLES = 650_000
beats["edge"] = (beats["sample"] - PRE < 0) | (beats["sample"] + POST > N_SAMPLES)
edge = beats[beats["edge"]]
print("por split:", edge.groupby("split").size().reindex(list(SPLITS), fill_value=0).to_dict())
print("por registro:", edge.groupby("record").size().to_dict())
check("latidos descartados por borde", len(edge), 39)

**Las dos decisiones de esta etapa, frente a su alternativa.**

- **D-016, criterio de `mlii_noisy`.** Se decidió marcar el latido si cualquier muestra de la ventana toca un tramo ruidoso, porque el autoencoder ve la ventana entera. Mirar solo la muestra del R marcaría algunos latidos menos: los que están en el límite de un tramo.
- **D-017, cómo se calcula el RR.** En esta base todas las anotaciones de latido son los 15 símbolos AAMI (los demás no son QRS), así que "solo latidos AAMI" y "todos los QRS" coinciden. Se decidió calcularlo antes de quitar los bordes, para que los vecinos de un latido de borde conserven su RR real. Calcularlo después les dejaría `NaN` o un RR que salta un latido.

In [ ]:
noise_cmp = []
for r in RECORDS:
    rec_ann = ann[ann["record"] == r]
    r_s = rec_ann.loc[rec_ann["symbol"].isin(config.AAMI_MAP), "sample"].to_numpy()
    channel = headers[r].index(config.LEAD)
    noise_cmp.append((SPLIT_OF[r], mlii_noise_mask(rec_ann, r_s, channel).sum(),
                      mlii_noise_mask_window(rec_ann, r_s, channel).sum()))
noise_cmp = pd.DataFrame(noise_cmp, columns=["split", "por R", "por ventana"])
display(noise_cmp.groupby("split").sum().reindex(list(SPLITS)))

changed = 0
for r, g in beats.groupby("record"):
    kept = ~g["edge"].to_numpy()
    before = np.column_stack(rr_intervals(g["sample"].to_numpy()))[kept]
    after = np.column_stack(rr_intervals(g["sample"].to_numpy()[kept]))
    changed += int((~np.isclose(before, after, equal_nan=True)).any(axis=1).sum())
print(f"D-017: latidos cuyo RR cambia si se calcula después de quitar bordes: {changed}")

## Etapa 5 · Ensamblar C1

Un pipeline es una secuencia fija de pasos, sin nada manual en medio: **leer → filtrar → segmentar → juntar → validar → guardar**. Se corre siempre desde cero.

- **Tipos del contrato**: `X`, `rr_prev` y `rr_next` en `float32`; `symbol` y `aami` en `<U1`; `label` en `int8`; `record` en `int16`; `split` en `<U5`; `r_sample` en `int32`; `mlii_noisy` en `bool`. Nada puede quedar como `object`, porque `load_beats` abre el archivo con `allow_pickle=False`.
- **Sin normalizar**: C1 va en mV filtrados. El z-score por latido lo aplica Modelo (D-008). Conservar las unidades físicas permite además el experimento pendiente sin normalización.
- **Procedencia**: `meta.json` guarda la versión, el commit, los parámetros y los conteos split × AAMI. Si alguien mueve un registro de split o cambia el filtro, se nota ahí.
- **Orden estable**: registros en orden ascendente y latidos en orden temporal, para que dos ejecuciones den arrays idénticos.

In [ ]:
C1_DTYPES = {
    "X": np.float32, "symbol": "<U1", "aami": "<U1", "label": np.int8, "record": np.int16,
    "split": "<U5", "r_sample": np.int32, "rr_prev": np.float32, "rr_next": np.float32,
    "mlii_noisy": np.bool_,
}
def process_record(record: int) -> dict[str, np.ndarray]:
    """Un registro → sus filas de C1 (antes de convertir a los tipos del contrato)."""
    x, channel = load_mlii(record)
    y = bandpass(x)  # registro completo, antes de recortar (D-007)
    rec_ann = read_annotations(record)
    b = rec_ann[rec_ann["symbol"].isin(config.AAMI_MAP)]  # D-014
    r, symbols = b["sample"].to_numpy(), b["symbol"].to_numpy()
    windows, keep = extract_windows(y, r)
    if RR_BEFORE_EDGE_DROP:  # D-017
        rr_prev, rr_next = (v[keep] for v in rr_intervals(r))
    else:
        rr_prev, rr_next = rr_intervals(r[keep])
    sym = symbols[keep]
    n = len(sym)
    return {
        "X": windows,
        "symbol": sym,
        "aami": np.array([config.AAMI_MAP[s] for s in sym]),
        "label": ~np.isin(sym, list(config.NORMAL_SYMBOLS)),
        "record": np.full(n, record),
        "split": np.full(n, SPLIT_OF[record]),
        "r_sample": r[keep],
        "rr_prev": rr_prev,
        "rr_next": rr_next,
        "mlii_noisy": NOISE_FN(rec_ann, r, channel)[keep],  # D-016
    }


def build_c1(records) -> dict[str, np.ndarray]:
    """Procesa los registros en el orden dado y convierte cada columna al tipo del contrato."""
    parts = [process_record(r) for r in records]
    return {k: np.concatenate([p[k] for p in parts]).astype(dt) for k, dt in C1_DTYPES.items()}

**Construir y validar.** `validate_beats` (de `ecg.io`) revisa claves, formas, tipos y que `label` sea consistente con `symbol`. Si algo falla, lanza un error y no se llega a guardar nada.

In [ ]:
c1 = build_c1(RECORDS)
validate_beats(c1)
display(pd.DataFrame({k: [v.shape, v.dtype.str] for k, v in c1.items()}, index=["forma", "dtype"]))

In [ ]:
n_rows = pd.Series(c1["split"]).value_counts()
for s in SPLITS:
    expected = int(((beats["split"] == s) & ~beats["edge"]).sum())
    check(f"filas de {s} = latidos AAMI − bordes", int(n_rows[s]), expected)
check("filas totales (100 733 − 39)", len(c1["X"]), 100_694)
check("registros en C1", np.unique(c1["record"]).tolist(), RECORDS)

rec_split = pd.DataFrame({"record": c1["record"], "split": c1["split"]}).drop_duplicates()
check("cada registro aparece en un solo split", int(rec_split["record"].duplicated().sum()), 0)
check("el split de cada registro es el de config.py",
      all(SPLIT_OF[r] == s for r, s in rec_split.itertuples(index=False)), True)
check("label = 0 ⇔ symbol = 'N'", bool(np.array_equal(c1["label"] == 0, c1["symbol"] == "N")), True)
check("X sin NaN ni infinitos", bool(np.isfinite(c1["X"]).all()), True)
check("RR sin NaN fuera del primer o último latido de cada registro",
      int(np.isnan(c1["rr_prev"]).sum() + np.isnan(c1["rr_next"]).sum()) <= 2 * len(RECORDS), True)

**Las dos filas de referencia.** Así quedan en C1 el N 8178 y el V 8377 del registro 119, columna por columna, y su `X` es exactamente la ventana de la señal filtrada.

In [ ]:
idx = [int(np.flatnonzero((c1["record"] == 119) & (c1["r_sample"] == s))[0]) for s in (8178, 8377)]
rows = pd.DataFrame({k: c1[k][idx] for k in C1_DTYPES if k != "X"})
display(rows)

got = rows.assign(rr_prev=rows["rr_prev"].astype(float).round(3),
                  rr_next=rows["rr_next"].astype(float).round(3)).to_dict("records")
expected = [
    {"symbol": "N", "aami": "N", "label": 0, "record": 119, "split": "val", "r_sample": 8178,
     "rr_prev": 1.303, "rr_next": 0.553, "mlii_noisy": False},
    {"symbol": "V", "aami": "V", "label": 1, "record": 119, "split": "val", "r_sample": 8377,
     "rr_prev": 0.553, "rr_next": 1.342, "mlii_noisy": False},
]
check("filas de referencia del 119", got, expected)
window_8178 = y119[8178 - PRE : 8178 + POST].astype(np.float32)
check("X del N 8178 = ventana de la señal filtrada",
      bool(np.array_equal(c1["X"][idx[0]], window_8178)), True)

**Reproducibilidad.** Se vuelven a procesar dos registros desde cero y se comparan con lo ya construido. La huella SHA-256 de `X` sirve para comparar entre máquinas: si dos personas generan C1 con el mismo código y los mismos datos, debe coincidir (salvo diferencias de versión de SciPy o NumPy).

In [ ]:
again = build_c1([108, 119])
first = {k: v[np.isin(c1["record"], [108, 119])] for k, v in c1.items()}
same = all(np.array_equal(first[k], again[k], equal_nan=first[k].dtype.kind == "f") for k in c1)
check("dos ejecuciones dan arrays idénticos (108 y 119)", same, True)
print("SHA-256 de X:", hashlib.sha256(c1["X"].tobytes()).hexdigest()[:16], "…")

**Guardar.** Primero se arma `meta.json` con los campos del contrato. Solo se guarda si no hay discrepancias bloqueantes; las diferencias con la literatura no bloquean, pero quedan en el resumen final. `data/processed/` está en `.gitignore`: estos archivos nunca van al repositorio.

In [ ]:
def current_commit() -> str:
    """Hash del commit actual; avisa si hay cambios sin commitear."""
    def git(*args):
        return subprocess.run(["git", *args], cwd=REPO_ROOT, capture_output=True, text=True,
                              check=True).stdout.strip()
    try:
        commit, dirty = git("rev-parse", "HEAD"), git("status", "--porcelain")
    except (OSError, subprocess.CalledProcessError):
        return "desconocido"
    if dirty:
        print("Aviso: hay cambios sin commitear; el commit no describe exactamente este código.")
    return commit


counts_c1 = pd.crosstab(c1["split"], c1["aami"], rownames=["split"], colnames=["aami"]).reindex(
    index=list(SPLITS), columns=AAMI_CLASSES, fill_value=0
)
meta = {
    "data_version": DATA_VERSION,
    "git_commit": current_commit(),
    "fs": FS,
    "window": {"pre": PRE, "post": POST, "total": WIN},
    "filter": {"low": config.BANDPASS[0], "high": config.BANDPASS[1],
               "order": config.FILTER_ORDER},
    "counts": {s: {c: int(counts_c1.loc[s, c]) for c in AAMI_CLASSES} for s in SPLITS},
    "created_at": datetime.now(UTC).isoformat(timespec="seconds"),
}

blocking = [d for d in DISCREPANCIES if d["bloquea"]]
assert not blocking, f"No se guarda C1: hay discrepancias bloqueantes {blocking}"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
np.savez_compressed(PROCESSED_DIR / "beats.npz", **c1)
with open(PROCESSED_DIR / "meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print(json.dumps(meta, indent=2, ensure_ascii=False))

**Leerlo como lo leerán Modelo y Evaluación.** `load_beats` devuelve train y val por defecto y bloquea test sin `allow_test=True`.

In [ ]:
dev = load_beats(PROCESSED_DIR / "beats.npz")
validate_beats(dev)
print(f"load_beats → {len(dev['X']):,} filas · splits {sorted(set(dev['split']))}")
try:
    load_beats(PROCESSED_DIR / "beats.npz", splits=("test",))
except PermissionError:
    print("test bloqueado sin allow_test=True, como corresponde")

### Reporte (para el informe)

Conteos finales de C1 por split y clase, con el porcentaje de latidos con MLII ruidosa. Las figuras que siguen usan solo train y val.

In [ ]:
report = counts_c1.copy()
report["total"] = report.sum(axis=1)
report["% mlii_noisy"] = (
    pd.Series(c1["mlii_noisy"]).groupby(c1["split"]).mean().mul(100).round(2).reindex(list(SPLITS))
)
display(report)

**Galería por símbolo (train + val).** Hasta 40 latidos de cada símbolo, elegidos a intervalos regulares, con su media. Sirve para ver de un vistazo qué formas tendrá que distinguir el autoencoder y cuáles tienen tan pocos ejemplos que sus métricas serán poco fiables.

In [ ]:
symbols_dev = pd.Series(dev["symbol"]).value_counts()
order = sorted(symbols_dev.index,  # por clase AAMI y, dentro de cada clase, del más frecuente
               key=lambda s: (AAMI_CLASSES.index(config.AAMI_MAP[s]), -symbols_dev[s]))
n_cols = 4
n_rows_fig = int(np.ceil(len(order) / n_cols))
fig, axes = plt.subplots(n_rows_fig, n_cols, figsize=(13, 2.6 * n_rows_fig), sharex=True,
                         sharey=True)
for ax, sym in zip(axes.flat, order):
    rows_sym = np.flatnonzero(dev["symbol"] == sym)
    pick = rows_sym[np.linspace(0, len(rows_sym) - 1, min(40, len(rows_sym))).astype(int)]
    ax.plot(t_ms, dev["X"][pick].T, color=C_MAIN, alpha=0.15, lw=0.7)
    ax.plot(t_ms, dev["X"][rows_sym].mean(axis=0), color=INK, lw=1.6)
    ax.set_title(f"{sym} · AAMI {config.AAMI_MAP[sym]} · n = {len(rows_sym):,}", fontsize=9)
for i, ax in enumerate(axes.flat):
    if i >= len(order):
        ax.set_visible(False)
    elif i + n_cols >= len(order):  # sin panel debajo: lleva su propio eje x
        ax.tick_params(labelbottom=True)
        ax.set_xlabel("ms desde la anotación")
fig.suptitle("Train + val · latidos por símbolo (línea negra: media de todos)", y=1.01)
fig.tight_layout()
plt.show()

La galería también muestra cuánto pesa el paciente. Los L de train vienen de dos registros con QRS de polaridad opuesta: positivo en el 109 y negativo en el 207. Su media no se parece a ninguno de los dos. Es la misma razón por la que el split se hace por paciente y no por latido.

## Decisiones de datos y discrepancias

Las decisiones de la etapa de datos, cerradas el 16 oct 2026 y registradas en `docs/decisions.md`, con la evidencia que aporta este notebook.

| | Pregunta | Decisión | Evidencia (ver etapa) |
|---|---|---|---|
| D-014 | ¿Qué anotaciones son latidos? | Los 15 símbolos de `AAMI_MAP`; fuera `! \| x " [ ] + ~` | 3 153 no-latidos, ninguno es un QRS clasificable (2) |
| D-015 | ¿Una sola fuente para los parámetros? | `config.py`; se eliminó `configs/data.yaml` | Nadie leía el YAML (Preparación) |
| D-016 | ¿Cuándo es ruidoso un latido? | Si cualquier muestra de la ventana cae en un tramo `~`; −1 cuenta | Comparación por R contra por ventana (4) |
| D-017 | ¿RR antes o después de quitar bordes? | Antes, entre latidos AAMI consecutivos | 39 latidos cambiarían su RR (4) |
| D-018 | ¿"Normal" = símbolo N o N sinusal? | Símbolo N en cualquier ritmo | 14,6 % de los N de train en FA, flutter o preexcitación (2) |
| D-019 | ¿Dónde vive el pipeline? | Este notebook documenta; el código pasa a `src/ecg/data/` con tests | — |

Abajo, el resumen de todo lo que no coincidió con lo esperado.

In [ ]:
if DISCREPANCIES:
    display(pd.DataFrame(DISCREPANCIES))
else:
    print("Sin discrepancias.")